# v2 — Step 05: Cell Type Annotation

## Purpose
Identify cell types in each Leiden cluster using:
1. Cluster-level differential expression (Wilcoxon rank-sum test)
2. Manual comparison against published marker gene panels
3. Visualisation (dotplots, heatmaps, UMAP overlays)

## Marker gene panels
From Parikh et al. (2019, *Cell*) and Smillie et al. (2019, *Cell*):

| Cell type | Markers |
|-----------|--------|
| Colonocytes (absorptive enterocytes) | FABP1, SLC26A3, CA1, CA2, TMEM37 |
| BEST4+ enterocytes | BEST4, OTOP2, CA7 |
| Goblet cells | MUC2, TFF3, FCGBP, ZG16 |
| Enteroendocrine | CHGA, NEUROD1, TPH1 |
| Stem / TA progenitor | LGR5, OLFM4, ASCL2 |
| Proliferating | MKI67, TOP2A |
| Inflammation-associated | LCN2, DUOX2, DUOXA2, REG1A, S100A8, S100A9 |
| Immune contamination (T cell) | CD3D, CD3E |
| Immune contamination (mast cell) | TPSAB1, TPSB2 |
| Antigen presentation / tuft | HLA-DRA, HLA-DPB1, CD74 |

## Input
- `data/processed/v2/GSE116222_v2_clustered.h5ad`

## Output
- `data/processed/v2/GSE116222_v2_annotated.h5ad`
- `results/v2_cluster_markers.csv`
- `results/v2_cell_type_composition.csv`

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.set_figure_params(dpi=150, facecolor='white', frameon=False)
os.makedirs('results', exist_ok=True)

LEIDEN_KEY        = 'leiden_1.0'
CONDITION_ORDER   = ['Healthy', 'UC_noninflamed', 'UC_inflamed']
CONDITION_PALETTE = {'Healthy': '#4CAF50', 'UC_noninflamed': '#FF9800', 'UC_inflamed': '#F44336'}

In [ ]:
adata = sc.read_h5ad('data/processed/v2/GSE116222_v2_clustered.h5ad')
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(f"Clusters (leiden 1.0): {sorted(adata.obs[LEIDEN_KEY].unique().tolist())}")

## Cluster-level differential expression (Wilcoxon rank-sum test)

For each cluster, we find genes that are significantly more expressed in that cluster
versus all other cells combined. The Wilcoxon rank-sum test is non-parametric and
does not assume a particular expression distribution — appropriate for sparse count data.

We use `use_raw=True` so marker discovery is based on log-normalised values from
the full gene set (stored in `adata.raw`), not the scaled HVG-only matrix.

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby=LEIDEN_KEY,
    method='wilcoxon',
    use_raw=True,
    pts=True  # include fraction of cells expressing each gene
)
print("rank_genes_groups complete.")

In [ ]:
# Extract top 10 markers per cluster into a DataFrame
markers_df = sc.get.rank_genes_groups_df(adata, group=None, key='rank_genes_groups')

# Save to CSV
markers_df.to_csv('results/v2_cluster_markers.csv', index=False)
print(f"Saved: results/v2_cluster_markers.csv  ({len(markers_df):,} rows)")

# Preview top 5 per cluster
top5 = (
    markers_df
    .groupby('group')
    .apply(lambda x: x.nlargest(5, 'scores'))
    .reset_index(drop=True)
)
print("\nTop 5 markers per cluster (by Wilcoxon score):")
print(top5[['group', 'names', 'scores', 'logfoldchanges', 'pvals_adj']].to_string(index=False))

In [ ]:
# Ranked gene plot — top 5 markers per cluster shown as a panel
sc.pl.rank_genes_groups(
    adata, n_genes=5,
    key='rank_genes_groups',
    show=False
)
plt.suptitle('Top 5 marker genes per cluster (Wilcoxon)', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('figures/v2/05_marker_genes_ranked.png', dpi=150, bbox_inches='tight')
plt.show()

## Define marker gene panels

Marker panels from Parikh et al. (2019) and Smillie et al. (2019).
Only include genes present in the dataset (some lowly expressed markers may have
been removed by QC gene filtering).

In [ ]:
marker_panels = {
    'Colonocytes':       ['FABP1', 'SLC26A3', 'CA1', 'CA2', 'TMEM37'],
    'BEST4+ enterocytes':['BEST4', 'OTOP2', 'CA7'],
    'Goblet':            ['MUC2', 'TFF3', 'FCGBP', 'ZG16'],
    'Enteroendocrine':   ['CHGA', 'NEUROD1', 'TPH1'],
    'Stem / TA':         ['LGR5', 'OLFM4', 'ASCL2'],
    'Proliferating':     ['MKI67', 'TOP2A'],
    'Inflammation':      ['LCN2', 'DUOX2', 'DUOXA2', 'REG1A', 'S100A8', 'S100A9'],
    'T cells (contam)':  ['CD3D', 'CD3E'],
    'Mast (contam)':     ['TPSAB1', 'TPSB2'],
    'Antigen pres.':     ['HLA-DRA', 'HLA-DPB1', 'CD74']
}

# Check which markers are present in the raw gene set
raw_genes = set(adata.raw.var_names)
for panel, genes in marker_panels.items():
    missing = [g for g in genes if g not in raw_genes]
    present = [g for g in genes if g in raw_genes]
    print(f"{panel:25s} present: {present}  MISSING: {missing if missing else 'none'}")

## Dotplot — all marker panels by cluster

The dotplot shows two things simultaneously:
- **Colour intensity** = mean expression level in the cluster
- **Dot size** = fraction of cells in the cluster expressing the gene

A useful marker has high expression AND high fraction in its target cluster,
and low values in all others.

In [ ]:
# Build a flat list of all markers, filter to those present
all_markers = [
    g for genes in marker_panels.values()
    for g in genes
    if g in raw_genes
]

sc.pl.dotplot(
    adata,
    var_names=marker_panels,
    groupby=LEIDEN_KEY,
    use_raw=True,
    standard_scale='var',
    show=False
)
plt.suptitle('Marker gene expression by cluster', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('figures/v2/05_marker_dotplot.png', dpi=150, bbox_inches='tight')
plt.show()

## UMAP overlays — absorptive lineage markers

Colonocytes are the most abundant cell type in healthy colon. Key markers:
- **FABP1** — fatty acid binding protein, mature colonocyte
- **SLC26A3** — chloride/bicarbonate transporter, highly absorptive colonocyte
- **CA2** — carbonic anhydrase, surface colonocyte

In [ ]:
colonocyte_markers = [g for g in ['FABP1', 'SLC26A3', 'CA1', 'CA2', 'TMEM37'] if g in raw_genes]
sc.pl.umap(
    adata, color=colonocyte_markers,
    use_raw=True, ncols=3,
    title=[f'{g} (colonocyte)' for g in colonocyte_markers],
    show=False
)
plt.suptitle('Absorptive colonocyte markers on UMAP', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('figures/v2/05_umap_colonocyte_markers.png', dpi=150, bbox_inches='tight')
plt.show()

## UMAP overlays — BEST4+ enterocyte markers

BEST4+ enterocytes are a rare, recently characterised colonocyte subpopulation
identified in Parikh et al. (2019). They are markedly depleted in UC-inflamed tissue.

In [ ]:
best4_markers = [g for g in ['BEST4', 'OTOP2', 'CA7'] if g in raw_genes]
sc.pl.umap(
    adata, color=best4_markers,
    use_raw=True, ncols=3,
    title=[f'{g} (BEST4+)' for g in best4_markers],
    show=False
)
plt.suptitle('BEST4+ enterocyte markers on UMAP', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('figures/v2/05_umap_best4_markers.png', dpi=150, bbox_inches='tight')
plt.show()

## UMAP overlays — secretory lineage (goblet & enteroendocrine)

In [ ]:
secretory_markers = [g for g in ['MUC2', 'TFF3', 'FCGBP', 'ZG16', 'CHGA', 'NEUROD1'] if g in raw_genes]
sc.pl.umap(
    adata, color=secretory_markers,
    use_raw=True, ncols=3,
    title=[f'{g}' for g in secretory_markers],
    show=False
)
plt.suptitle('Secretory lineage markers on UMAP (goblet / enteroendocrine)', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('figures/v2/05_umap_secretory_markers.png', dpi=150, bbox_inches='tight')
plt.show()

## UMAP overlays — stem / progenitor and proliferating

In [ ]:
stem_markers = [g for g in ['LGR5', 'OLFM4', 'ASCL2', 'MKI67', 'TOP2A'] if g in raw_genes]
sc.pl.umap(
    adata, color=stem_markers,
    use_raw=True, ncols=3,
    title=[f'{g}' for g in stem_markers],
    show=False
)
plt.suptitle('Stem / progenitor and proliferating cell markers on UMAP', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('figures/v2/05_umap_stem_proliferating_markers.png', dpi=150, bbox_inches='tight')
plt.show()

## UMAP overlays — inflammation-associated markers

These genes are upregulated in stressed or inflamed epithelial cells:
- **LCN2** — lipocalin-2, acute-phase protein; strong UC inflamed marker
- **DUOX2/DUOXA2** — dual oxidase system; antimicrobial ROS production
- **REG1A** — regenerating protein; epithelial repair and inflammation

In [ ]:
inflam_markers = [g for g in ['LCN2', 'DUOX2', 'DUOXA2', 'REG1A', 'S100A8', 'S100A9'] if g in raw_genes]
sc.pl.umap(
    adata, color=inflam_markers,
    use_raw=True, ncols=3,
    title=[f'{g} (inflammation)' for g in inflam_markers],
    show=False
)
plt.suptitle('Inflammation-associated markers on UMAP', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('figures/v2/05_umap_inflammation_markers.png', dpi=150, bbox_inches='tight')
plt.show()

## UMAP overlays — immune contamination markers

EPCAM+ sorted samples can contain small numbers of immune cells.
Clusters enriched for T cell or mast cell markers should be flagged as
contamination and either removed or annotated separately.

In [ ]:
immune_markers = [g for g in ['CD3D', 'CD3E', 'TPSAB1', 'TPSB2', 'HLA-DRA', 'CD74'] if g in raw_genes]
sc.pl.umap(
    adata, color=immune_markers,
    use_raw=True, ncols=3,
    title=[f'{g}' for g in immune_markers],
    show=False
)
plt.suptitle('Immune contamination markers on UMAP', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('figures/v2/05_umap_immune_markers.png', dpi=150, bbox_inches='tight')
plt.show()

## Manual cell type annotation

Based on the dotplot, UMAP overlays, and top Wilcoxon markers above, assign a cell
type label to each cluster. Update the dictionary below after reviewing the plots.

**Annotation rules**:
- A cluster requires ≥2 concordant canonical markers at high expression and fraction
- A cluster dominated by a single condition is noted but not renamed (condition-enrichment
  is a biological finding, not a cell type definition)
- Clusters with immune markers as top DE genes → annotated as immune contamination
- Ambiguous clusters → labelled 'Unknown' pending further marker investigation

**Edit the dictionary below after reviewing the figures.**

In [ ]:
# ----------------------------------------------------------------
# EDIT THIS DICTIONARY after reviewing figures above.
# Keys = Leiden cluster numbers (as strings), Values = cell type labels.
# This is intentionally left as a placeholder — annotations must be
# made by inspecting the dotplot and UMAP marker overlays.
# ----------------------------------------------------------------

cluster_annotation = {
    # Example entries — replace with evidence-based assignments:
    # '0':  'Colonocytes',
    # '1':  'Colonocytes',
    # '2':  'Goblet',
    # ...
}

# Get all cluster IDs
all_clusters = sorted(adata.obs[LEIDEN_KEY].unique().tolist(), key=int)
print(f"Clusters to annotate: {all_clusters}")
print(f"Annotations provided: {list(cluster_annotation.keys())}")
unannotated = [c for c in all_clusters if c not in cluster_annotation]
if unannotated:
    print(f"\nClusters without annotation yet: {unannotated}")
    print("Review figures above and fill in cluster_annotation dict, then re-run from this cell.")

In [ ]:
# Apply annotations (run after filling the dictionary above)
if len(cluster_annotation) == len(all_clusters):
    adata.obs['cell_type'] = (
        adata.obs[LEIDEN_KEY]
        .map(cluster_annotation)
        .astype('category')
    )
    print("Cell type annotations applied:")
    print(adata.obs['cell_type'].value_counts().to_string())
else:
    print(f"Incomplete annotation — {len(all_clusters) - len(cluster_annotation)} clusters still need labels.")
    print("Assign all clusters before running the visualisation cells below.")

In [ ]:
# UMAP coloured by annotated cell type
# (run after cell_type column is populated)
if 'cell_type' in adata.obs.columns:
    sc.pl.umap(
        adata, color='cell_type',
        legend_loc='right margin',
        title='Cell type annotations (v2, Harmony-corrected, raw counts)',
        show=False
    )
    plt.tight_layout()
    plt.savefig('figures/v2/05_umap_cell_types.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("cell_type column not yet created — annotate clusters first.")

## Cell type composition by condition

Compare how cell type proportions change across Healthy, UC non-inflamed, and UC inflamed.
Key expected findings (Parikh et al. 2019):
- BEST4+ enterocytes: depleted in UC inflamed
- Inflammation-associated colonocytes: enriched in UC inflamed
- Goblet cells: generally reduced in inflamed

In [ ]:
if 'cell_type' in adata.obs.columns:
    comp = (
        adata.obs
        .groupby(['condition', 'cell_type'])
        .size()
        .unstack(fill_value=0)
    )
    comp_frac = comp.div(comp.sum(axis=1), axis=0)
    comp_frac = comp_frac.loc[CONDITION_ORDER]

    fig, ax = plt.subplots(figsize=(12, 5))
    comp_frac.T.plot(
        kind='bar', stacked=False,
        color=[CONDITION_PALETTE[c] for c in CONDITION_ORDER],
        ax=ax, edgecolor='white', linewidth=0.3, width=0.7
    )
    ax.set_xlabel('Cell type')
    ax.set_ylabel('Fraction of cells in condition')
    ax.set_title('Cell type composition by condition (v2 analysis)')
    ax.legend(title='Condition', bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig('figures/v2/05_cell_type_composition_by_condition.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Save to CSV
    comp_frac.to_csv('results/v2_cell_type_composition.csv')
    print("Saved: results/v2_cell_type_composition.csv")
else:
    print("Annotate clusters first.")

## Save annotated AnnData

In [ ]:
out_path = 'data/processed/v2/GSE116222_v2_annotated.h5ad'
adata.write_h5ad(out_path)
print(f"Saved : {out_path}")
print(f"Size  : {os.path.getsize(out_path) / 1e6:.1f} MB")
print(f"Shape : {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(f"\n{adata}")